# Tema 3 · Entrenamiento, prueba y validación cruzada

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

En el **CESFAM Los Volcanes** de Chillán Viejo, casi **4 de cada 10 pacientes no llegan** a su hora agendada, y esas horas se pierden. El director quiere llamar por teléfono, el día antes, a los pacientes con más riesgo de faltar. Un proveedor le ofreció un modelo «con 100 % de acierto», y el director no sabe si creerle. Les pide construir un modelo propio y, sobre todo, **decirle honestamente cuánto acierta**.

> **❓ La pregunta del cliente:** ¿Qué modelo conviene usar para anticipar quién faltará, y cuánto acierto se le puede prometer honestamente al director?

## 🗂️ Los datos: `t3_citas.csv` (1.200 citas con datos del paciente y de la hora, y si asistió)

| Columna | Qué significa |
|---|---|
| `cita` | número de la cita |
| `edad` | edad del paciente, en años |
| `dias_anticipacion` | días entre que se dio la hora y la cita |
| `recordatorio_sms` | Sí / No: si se le envió un recordatorio por SMS |
| `distancia_km` | distancia de la casa al CESFAM, en km |
| `inasistencias_previas` | veces que faltó a una hora en el último año |
| `tipo_atencion` | Control / Morbilidad / Dental |
| `asistio` | **Sí / No**: si el paciente llegó a la cita ← lo que se quiere predecir |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t3_citas.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 1200 filas y 8 columnas.

## 1. Conocer los datos

📘 Cada fila es una **cita** del CESFAM: la edad del paciente, con cuántos días de anticipación se dio la hora, si
se le mandó un recordatorio por SMS, a qué distancia vive, cuántas veces faltó antes y el tipo de atención. La
columna `asistio` dice si el paciente llegó (**Sí**) o faltó (**No**).

In [ ]:
datos.head()

In [ ]:
conteo = datos["asistio"].value_counts()
print("Asistieron:", conteo["Sí"], "· Faltaron:", conteo["No"])
# ✅ Debe mostrar: Asistieron: 759 · Faltaron: 441

## 2. Preparar los datos y separar entrenamiento y prueba

📘 El árbol de scikit-learn solo trabaja con números. `pd.get_dummies()` convierte cada columna de texto en
columnas de 0 y 1 (por ejemplo, `tipo_atencion` pasa a `tipo_atencion_Control`, `tipo_atencion_Dental` y
`tipo_atencion_Morbilidad`).

Luego viene la idea central del tema: **el modelo se evalúa con datos que no vio**. Se separa el 75 % de las citas
para **entrenar** (aprender) y el 25 % para **probar** (el «examen»).

In [ ]:
X = pd.get_dummies(datos.drop(columns=["cita", "asistio"]), dtype=int)
y = datos["asistio"]
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.25, random_state=2026, stratify=y)
print("Columnas:", X.shape[1], "· Entrenamiento:", len(X_ent), "citas · Prueba:", len(X_pru), "citas")
# ✅ Debe mostrar: Columnas: 9 · Entrenamiento: 900 citas · Prueba: 300 citas

## 3. La línea base: ZeroR

📘 ¿Cuánto se acierta sin modelo, diciendo siempre la respuesta más común («Sí asiste»)?

In [ ]:
acierto_zeror = (y_pru == "Sí").mean()
print(f"ZeroR (decir siempre 'Sí asiste'): {acierto_zeror:.0%}")
# ✅ Debe mostrar: ZeroR (decir siempre 'Sí asiste'): 63%

## 4. Un árbol sin límites

📘 Un árbol de decisión sin límite de profundidad sigue haciendo preguntas hasta que **cada hoja queda pura**.
Se mide su acierto **dos veces**: con las citas con que aprendió (entrenamiento) y con las que no vio (prueba).

In [ ]:
arbol_libre = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_ent, y_ent)
print(f"Árbol sin límite → entrenamiento: {arbol_libre.score(X_ent, y_ent):.0%} · prueba: {arbol_libre.score(X_pru, y_pru):.0%}")
print("Profundidad que alcanzó:", arbol_libre.get_depth(), "· Hojas:", arbol_libre.get_n_leaves())
# ✅ Debe mostrar: Árbol sin límite → entrenamiento: 100% · prueba: 63%  /  Profundidad que alcanzó: 15 · Hojas: 215

### ✍️ El árbol acierta el 100 % de las citas de entrenamiento. ¿Cuánto acierta en prueba y cómo se compara con ZeroR? ¿Qué le pasó al árbol? Usen la palabra **sobreajuste** y una comparación con un estudiante.

_Su respuesta:_

## 5. Un árbol más simple

📘 `max_depth=3` limita el árbol a tres preguntas seguidas. Aprende menos detalles, pero más generales.

In [ ]:
arbol3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_ent, y_ent)
print(f"Árbol de profundidad 3 → entrenamiento: {arbol3.score(X_ent, y_ent):.0%} · prueba: {arbol3.score(X_pru, y_pru):.0%}")
# ✅ Debe mostrar: Árbol de profundidad 3 → entrenamiento: 76% · prueba: 74%

## 6. Un solo examen puede engañar

📘 El acierto en prueba depende de **qué citas cayeron** en la prueba. La celda repite la separación 20 veces, con
20 sorteos distintos (`random_state` de 0 a 19), y entrena el **mismo** árbol de profundidad 3 cada vez.

In [ ]:
aciertos = []
for semilla in range(20):
    Xe, Xp, ye, yp = train_test_split(X, y, test_size=0.25, random_state=semilla, stratify=y)
    aciertos.append(DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xe, ye).score(Xp, yp))
print("Aciertos en prueba:", " · ".join(f"{a:.0%}" for a in aciertos))
print(f"El mismo modelo, con 20 separaciones distintas: entre {min(aciertos):.0%} y {max(aciertos):.0%}")
# ✅ Debe mostrar: Aciertos en prueba: 74% · 74% · 72% · 72% · 74% · 73% · 74% · 73% · 75% · 76% · 74% · 74% · 77% · 76% · 78% · 73% · 73% · 72% · 74% · 71%  /  El mismo modelo, con 20 separaciones distintas: entre 71% y 78%

### ✍️ El modelo es siempre el mismo. ¿Por qué su acierto cambia según la separación? Si el CESFAM hubiera hecho una sola separación «con mala suerte», ¿qué número le habrían informado al director?

_Su respuesta:_

## 7. Validación cruzada

📘 La **validación cruzada de 5 pliegues** usa todos los datos de entrenamiento para evaluar, sin trampa:

1. Divide el entrenamiento en **5 partes** (pliegues).
2. Entrena con 4 partes y evalúa con la que quedó fuera.
3. Repite 5 veces, dejando fuera una parte distinta cada vez.
4. Informa el **promedio** de los 5 aciertos y **cuánto varían** (la desviación).

Cada cita se usa 4 veces para aprender y **una vez para evaluar**. `cross_val_score()` hace todo eso en una línea.

In [ ]:
cv = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=0), X_ent, y_ent, cv=5)
print("Acierto en cada pliegue:", " · ".join(f"{a:.0%}" for a in cv))
print(f"Promedio: {cv.mean():.0%} (± {cv.std():.0%})")
# ✅ Debe mostrar: Acierto en cada pliegue: 76% · 75% · 71% · 73% · 73%  /  Promedio: 74% (± 2%)

## 8. Elegir la profundidad con validación cruzada

📘 Ahora se usa la validación cruzada para **elegir** la profundidad del árbol. Prueben cada profundidad y anoten
los tres aciertos. (`None` significa sin límite.)

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
PROFUNDIDAD = 3        # ✏️ pruebe con 1, 2, 3, 4, 5, 6, 8, 10 y None (al terminar, deje 3)

modelo = DecisionTreeClassifier(max_depth=PROFUNDIDAD, random_state=0).fit(X_ent, y_ent)
cv = cross_val_score(DecisionTreeClassifier(max_depth=PROFUNDIDAD, random_state=0), X_ent, y_ent, cv=5)
print(f"Profundidad {PROFUNDIDAD} → entrenamiento: {modelo.score(X_ent, y_ent):.0%} · "
      f"prueba: {modelo.score(X_pru, y_pru):.0%} · validación cruzada: {cv.mean():.0%}")

**✏️ Anoten aquí sus resultados:**

| Profundidad | Entrenamiento | Prueba | Validación cruzada (promedio) |
|:-:|:-:|:-:|:-:|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |
| 6 | | | |
| 8 | | | |
| 10 | | | |
| Sin límite | | | |

📘 Esta celda dibuja su tabla completa: las tres curvas para todas las profundidades.

In [ ]:
niveles = [1, 2, 3, 4, 5, 6, 8, 10, None]
ent, pru, val = [], [], []
for d in niveles:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_ent, y_ent)
    ent.append(m.score(X_ent, y_ent))
    pru.append(m.score(X_pru, y_pru))
    val.append(cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X_ent, y_ent, cv=5).mean())
etiquetas = [str(d) if d else "sin\nlímite" for d in niveles]
plt.figure(figsize=(7, 4))
plt.plot(etiquetas, ent, "o-", label="Entrenamiento")
plt.plot(etiquetas, val, "s-", label="Validación cruzada")
plt.plot(etiquetas, pru, "^--", label="Prueba")
plt.axhline(acierto_zeror, color="gray", ls=":", label="ZeroR")
plt.xlabel("Profundidad del árbol"); plt.ylabel("Acierto"); plt.legend()
plt.title("Más complejo no es mejor")
plt.show()

### ✍️ ¿Qué profundidad tiene el mejor promedio de validación cruzada? ¿Qué pasa con el acierto de entrenamiento y con el de validación cruzada a medida que el árbol crece? ¿Por qué se elige la profundidad con la validación cruzada y **no** con la prueba?

_Su respuesta:_

## 9. El modelo elegido, por dentro

📘 Un árbol de profundidad 3 se puede leer como reglas. `export_text()` lo muestra (`class: No` significa que
el árbol predice que el paciente **faltará**).

In [ ]:
print(export_text(arbol3, feature_names=list(X.columns)))

### ✍️ 🧠 **De la información al conocimiento.** El director del CESFAM quiere llamar por teléfono a los pacientes con más riesgo de faltar. ¿Qué modelo le recomiendan, qué acierto le pueden **prometer honestamente** (con qué número y por qué ese) y qué le dirían a un proveedor que le ofrece un modelo «con 100 % de acierto»?

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema3.ipynb`, junto con el informe en PDF.